# FactorCalculator 使用示例

本 Notebook 演示 `factor_calculator` 包的各种用法，包括解析单元规格、创建实例、以及使用计算器进行因子计算。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

## 1. 解析单元规格（Unit Specifications）

In [ ]:
from factor_calculator import parse_unit_spec

specs = [
    "KlineDMU(45)",
    "KlineDMU(interval=5)",
    "BiquotePEU(watching_time=60)",
    "PositionPnlDMU",
]

for spec in specs:
    class_name, params = parse_unit_spec(spec)
    print(f"{spec!r}")
    print(f"  -> class: {class_name}, params: {params!r}")

## 2. 列出可用的 DMU / PEU 类

> 需要安装 `rbt` 包才能运行此部分。

In [ ]:
from factor_calculator import get_available_classes

print("所有可用单元:")
for cls in get_available_classes():
    print(f"  - {cls}")

print("\nDMU 类:")
for cls in get_available_classes("DMU"):
    print(f"  - {cls}")

print("\nPEU 类:")
for cls in get_available_classes("PEU"):
    print(f"  - {cls}")

## 3. 创建单元实例

> 需要安装 `rbt` 包才能运行此部分。

In [ ]:
from factor_calculator import create_unit, create_units

# 创建单个单元
unit = create_unit("KlineDMU(interval=5)")
print(f"Created: {unit}")
print(f"  Name: {unit.name}")
print(f"  Interval: {unit.interval}")

In [ ]:
# 批量创建多个单元
units = create_units(["PositionPnlDMU", "TrendDMU"])
print(f"创建了 {len(units)} 个单元:")
for u in units:
    print(f"  - {u.name}")

## 4. SimpleFactorCalculator

简化版计算器，直接传入 DataFrame 计算单个 DMU/PEU，无需完整的 RBT Strategy 引擎。

In [ ]:
import pandas as pd
from factor_calculator import SimpleFactorCalculator

# 构造示例行情数据
md_data = pd.DataFrame({
    "name": pd.date_range("2024-03-15 09:30", periods=10, freq="1min"),
    "last_px": [100.0 + i * 0.1 for i in range(10)],
    "tot_sz": [100 * (i + 1) for i in range(10)],
    "oi": [1000 + i * 10 for i in range(10)],
})
md_data.set_index("name", inplace=True)
md_data

In [ ]:
# 使用 SimpleFactorCalculator 计算 DMU 因子
# 需要: rbt 已安装 + 有效的 db 目录

# calc = SimpleFactorCalculator(db_directory="/path/to/db")
# result = calc.calculate_dmu(
#     dmu_spec="PassThroughDMU",
#     contract="IF2403",
#     trade_date="2024-03-15",
#     md_data=md_data,
# )
# result

## 5. FactorCalculator（完整集成）

完整版计算器，集成 RBT Strategy 引擎、行情加载、结果数据库。

In [ ]:
from factor_calculator import FactorCalculator

# 初始化
# calc = FactorCalculator(
#     db_directory="/path/to/results",
#     md_directory="/path/to/market/data",
# )

# 执行因子计算
# result = calc.calculate(
#     units=[
#         "KlineDMU(interval=5)",
#         "BiquotePEU(watching_time=60)",
#     ],
#     load_factors=["KlineDMU__open", "KlineDMU__close"],
#     contract="IF2403",
#     trade_date="2024-03-15",
#     frequency="tick",
# )
# result

## 6. CLI 命令行用法

安装包后可直接在终端使用 `factor-calculator` 命令：

```bash
# 列出可用单元
factor-calculator list
factor-calculator list --dmu
factor-calculator list --peu

# 计算因子
factor-calculator calculate \
    --db /path/to/results \
    --md /path/to/market/data \
    --units "KlineDMU(5),BiquotePEU(60)" \
    --contract IF2403 \
    --date 2024-03-15

# 查看已有因子
factor-calculator factors \
    --db /path/to/results \
    --contract IF2403 \
    --date 2024-03-15
```